## Why Flights Look Curved on a Map (And How to Plot It in Python)

Have you ever looked at the flight map on a long-haul flight and wondered why your plane seems to be heading north instead of flying straight to its destination?

On a map, a flight from New York to London can look like a huge detour. It’s tempting to think the plane would get there faster by following a straight line. But it’s already taking the shortest route. The catch is that we’re looking at a spherical planet on a flat map, and that changes how the route appears.

In this post, we’ll see why maps can be misleading. With a few lines of Python in Google Colab — and no external datasets — we’ll calculate and plot flight paths to see what the shortest route actually looks like.

#### Great Circles and Rhumb Lines
Before we start coding, it helps to know about two kinds of routes.

A rhumb line is a route that keeps the same compass heading throughout the journey. On a Mercator map, it appears as a straight line. But on a spherical Earth, it’s usually longer than the shortest possible route.

That shortest route is called a great-circle route. It follows the curve of the Earth, so its direction gradually changes along the way. On a flat map, it often appears as a wide arc — which can make it look like a detour, even though it’s the shorter path.

Let’s use Python to calculate both routes and see how they compare.

In [8]:
# pip install pyproj

import folium
from pyproj import Geod

#### Step 1: Set up the Earth model and coordinates
Let’s use New York’s JFK Airport as our starting point and London Heathrow as our destination.

We’ll also tell Python which model of the Earth to use. For this, we’ll choose WGS84, the reference ellipsoid commonly used by GPS systems. It’s a more realistic model than a perfect sphere because it accounts for the Earth’s slight bulge around the equator.

In [9]:
# 1. Define Coordinates: New York (JFK) and London (LHR)
# Format: (latitude, longitude)
jfk = (40.6413, -73.7781)
lhr = (51.4700, -0.4543)

# 2. Initialize the WGS84 Reference Ellipsoid
geod = Geod(ellps='WGS84')

#### Step 2: Calculate the shortest route
If we simply connect the two airports on a flat map, we’ll get a straight line on the map — not the great-circle route we want to compare.

Instead, we’ll use pyproj to calculate points along the great-circle path between JFK and Heathrow. We’ll generate 100 points to give the route a smooth appearance, and calculate its distance in kilometers.

In [10]:
# 3. Calculate the Great Circle (Geodesic) intermediate points
# Note: pyproj expects (longitude, latitude) order!
great_circle_lon_lat = geod.npts(jfk[1], jfk[0], lhr[1], lhr[0], 100)

# Convert the results back to (latitude, longitude) for Folium mapping
great_circle_points = [(lat, lon) for lon, lat in great_circle_lon_lat]

# Prepend the start point and append the end point to complete the line
great_circle_points.insert(0, jfk)
great_circle_points.append(lhr)

# Calculate the actual geodesic distance for your article's trivia
az1, az2, distance_meters = geod.inv(jfk[1], jfk[0], lhr[1], lhr[0])
print(f"Shortest Flight Distance (Great Circle): {distance_meters / 1000:.0f} km")

Shortest Flight Distance (Great Circle): 5555 km


#### Step 3: Put the routes on the map
Now we can bring the results to life. We’ll use a gray Esri basemap so the routes are easy to see, then draw the rhumb line in red and the great-circle route in blue.

In [ ]:
import folium

# 4. Create the Interactive Map using CartoDB's built-in Light Gray theme
m = folium.Map(
    location=[46.0, -37.0], 
    zoom_start=3,
    tiles="CartoDB positron"  # Clean, minimal light gray background built right into Folium
)

# 5. Plot the Rhumb Line (Red, Dashed)
folium.PolyLine(
    locations=[jfk, lhr],
    color='red',
    weight=3,
    dash_array='10',
    tooltip='Rhumb Line (Looks shorter, but is actually longer!)'
).add_to(m)

# 6. Plot the Great Circle (Blue, Solid)
folium.PolyLine(
    locations=great_circle_points,
    color='blue',
    weight=4,
    tooltip='Great Circle (Shortest True Distance)'
).add_to(m)

# 7. Add Airport Markers
folium.Marker(jfk, tooltip='JFK Airport', icon=folium.Icon(color='black')).add_to(m)
folium.Marker(lhr, tooltip='LHR Airport', icon=folium.Icon(color='black')).add_to(m)

# 8. Display the map directly
m